# Run BEHAV3D Cellpose training

This notebook trains a Cellpose model for segmenting organoids, immune cells or other cell types populations in your own images.

### Overview

| Step | Description |
|------|-------------|
| 1 | Load packages |
| 2 | Select model type (organoid / immune cells-other cell types) |
| 3 | Set training & validation directories |
| 4 | Configure image information and channel labels |
| 5 | Train the Cellpose model |
| 6 | Visually validate predictions on the validation dataset|


### Requirements

Before running this notebook, make sure you have:

- A **training set** of 3D annotated images (~70% of your data) in `training_data/train/`
- A **validation set** of 3D annotated images (~30% of your data) in `training_data/val/`: these must be images the model has **never seen during training**
- Images in one of the supported formats: `png`, `jpg`, `jpeg`, `tif`, `tiff`
- Each image paired with a **mask file** in the same folder, sharing the same base filename but with a different suffix:
```
training_data/
├── train/
│   ├── sample01_img.tif    ← image
│   ├── sample01_masks.tif  ← mask
│   ├── sample02_img.tif
│   └── sample02_masks.tif
└── val/
    ├── sample03_img.tif
    └── sample03_masks.tif
```

> 💡 **Tip:** Run cells in order from top to bottom. Each step depends on the configuration set in the previous one.

### Load required packages

In [ ]:
#Load the packages
from behav3d.preprocessing.segmentation.cellpose_training import generate_2D_planes, train_cellpose
from cellpose import io, models
import napari
import pandas as pd
from pathlib import Path
import os
import torch
import zarr
import glob

from behav3d.widgets.utils import PathPicker


import ipywidgets as widgets
from IPython.display import display, clear_output


import random
random.seed(42)  # sets the seed for the random module

import numpy as np
np.random.seed(42)  # sets the seed for numpy random functions


%matplotlib inline

In [ ]:
# ---- App-wide state (lives in the kernel across cells) ----
import traitlets
from traitlets import Bool, Any

class _AppState(traitlets.HasTraits):
    # signals
    metadata_ready  = Bool(False)   # flips True when metadata is loaded
    segmentation_ready = Bool(False)
    tracking_ready  = Bool(False)
    # shared objects
    mdl     = Any(None)             # your MetadataLoader instance
    metadata= Any(None)

# reuse the same instance if you re-run this cell
APP = globals().get("APP") or _AppState()
globals()["APP"] = APP

# ---- tiny utility: run builder now or when a signal turns True ----
import ipywidgets as widgets

def render_when(container: widgets.Box, signal_name: str, builder):
    """
    If APP.<signal_name> is True, render immediately.
    Otherwise, register a one-shot observer that renders when it becomes True.
    `builder()` must return a widget (or a tuple/list of widgets).
    """
    # fast path: already ready
    if bool(getattr(APP, signal_name)):
        built = builder()
        container.children = built if isinstance(built, (tuple, list)) else (built,)
        return

    # otherwise, wait for the flip
    def _on(change):
        if change["name"] == signal_name and change["new"] is True:
            APP.unobserve(_on, names=[signal_name])  # one-shot
            built = builder()
            container.children = built if isinstance(built, (tuple, list)) else (built,)

    APP.observe(_on, names=[signal_name])

### Select the model that you want to train
Choose whether you want to train a model for **organoid** segmentation or for **immune cells / other cell types**.

This choice affects:
- The base Cellpose architecture used for training (`cyto3` for organoids, `nuclei` for immune cells/other cell types)
- The minimum object size used during inference

In [ ]:
# Selector for training on organoids or immune cells
options=['Organoid','Immune cells/Other cell types']
option_widget=widgets.Dropdown(
options=options,
value=options[0],
description='Do you want to train the Organoid or Immune cells/Other cell types model?',
style={'description_width': 'auto'},
layout={'width': 'max-content'}
)
display(option_widget)

### Set training directories
Specify the folders containing your **training** and **validation** images, and the filename substrings used to distinguish image files from mask files.

- **Train directory:** folder with training images and their masks
- **Validation directory:** folder with validation images and their masks for evaluating the model after training
- **Image identifier:** substring present in image filenames (e.g. `_img`)
- **Mask identifier:** substring present in mask filenames (e.g. `_masks`)

> ⚠️ Every image file must have a paired mask file with the same base name, differing only by the identifier substring.

In [ ]:
train_dir_picker = PathPicker(
    mode='dir',
    start_dir='.',
    default=r"training_data/train",
    description='Train directory:',
)
display(train_dir_picker)

val_dir_picker = PathPicker(
    mode='dir',
    start_dir='.',
    default=r"training_data/val",
    description='Validation directory:',
)
display(val_dir_picker)

#Filter
image_filter = widgets.Text(
    value=r"_img",
    description='Identifier for images',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='80%')
)
display(image_filter)

mask_filter = widgets.Text(
    description='Identifier for masks',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='80%')
)
if (option_widget.value=='Organoid'):
    mask_filter.value=r"_organoids"
    display(mask_filter)
elif (option_widget.value=='Immune cells/Other cell types'):
    mask_filter.value=r"_immune_other"
    display(mask_filter)

### Set data and channel configuration

Set the physical pixel sizes and dimension ordering of your images, then assign a **label** to each channel.

**Pixel sizes** are used to compute the anisotropy ratio (Z pixel size / XY pixel size), which tells Cellpose how to rescale the 3D volume correctly.

**Dimension order** describes how axes are arranged in your image files:
- `ZYX`: Z is the first axis (most common for tif stacks)
- `YXZ`: Z is the last axis

**Channel labels** are free-text names you assign to each channel (e.g. `tcell`, `organoid`, `dapi`). These labels are used later to select which channels to train.
Once you are happy with the configuration, click **Save configuration**.

In [ ]:
def make_channel_widgets(n):
    """Build the channel_widgets."""
    return [
        widgets.Text(
            value="tcell",
            description=f"Label for channel {i}",
            style={'description_width': 'auto'},
            layout=widgets.Layout(width='80%')
        )
        for i in range(n)
    ]

def rebuild_tab2():
    """Rebuild the Channel Config tab UI based on the channel_widgets."""
    channel_box = widgets.VBox(channel_widgets, layout=widgets.Layout(grid_gap='20px 25px'))
    button_box = widgets.VBox([save_button, change_button],layout=widgets.Layout(grid_gap = '20px 25px'))
    tab2_container.children = [
        widgets.HBox([channel_box, button_box],layout=widgets.Layout(grid_gap = '20px 25px'))
    ]
    
def on_channel_number_changed(change):
    if change["name"] == "value" and not channel_picker.disabled:
        new_n = change["new"]
        channel_widgets[:] = make_channel_widgets(new_n)
        rebuild_tab2()

# Widgets: Tab 1 ("Data config")

z_pixel_size = widgets.FloatText(
    description = 'Z pixel size (micrometers): ',
    value = 4.655,
    style={'description_width': 'auto'},
    layout = widgets.Layout(width='auto')
)
xy_pixel_size = widgets.FloatText(
    description = 'XY pixel size (micrometers): ',
    value = 1.17365196872,
    style={'description_width': 'auto'},
    layout = widgets.Layout(width='auto')
)
dim_orders=['ZYX','YXZ']
dim_order_picker = widgets.Dropdown(
    description = 'Dimension order: ',
    options=dim_orders,
    value=dim_orders[0],
    style={'description_width': 'auto'},
    layout={'width': 'max-content'}
)
channel_picker = widgets.BoundedIntText(
    description = 'Number of channels: ',
    value = 3,
    min=1,
    max=10,
    step=1,
    style={'description_width': 'auto'},
    layout = widgets.Layout(width='auto')
)

# Widgets: Tab 2 ("Channel config")

save_button = widgets.Button(description="Save configuration", button_style='success',layout={'width': 'max-content'})
change_button = widgets.Button(description="Change configuration", button_style='warning',layout={'width': 'max-content'}, disabled=True)
output_area = widgets.Output()
channel_widgets = make_channel_widgets(channel_picker.value)
tab2_container = widgets.VBox([])
rebuild_tab2()


def on_load_clicked(b):
    with output_area:
        global anisotropy, labels, labels_none, dim_order
        # Disable buttons
        for wid in [z_pixel_size, xy_pixel_size, dim_order_picker, channel_picker]:
            wid.disabled = True
        change_button.disabled=False
        # Disable buttons
        for cw in channel_widgets:
            cw.disabled=True
        # Load parameters
        anisotropy=z_pixel_size.value/xy_pixel_size.value
        dim_order=dim_order_picker.value
        labels=[cw.value for cw in channel_widgets]
        print(f"✅ Data and channel configuration saved, ready to go")

def on_change_clicked(b):
    with output_area:
        # Enable buttons
        for wid in [z_pixel_size, xy_pixel_size, dim_order_picker, channel_picker, save_button]:
            wid.disabled = False
        for cw in channel_widgets:
            cw.disabled=False
        change_button.disabled=True
        print(f"Waiting for new configuration...")

save_button.on_click(on_load_clicked)
change_button.on_click(on_change_clicked)
channel_picker.observe(on_channel_number_changed)

tab1 = widgets.VBox([z_pixel_size, xy_pixel_size, dim_order_picker, channel_picker],layout=widgets.Layout(grid_gap = '20px 25px'))
tab= widgets.VBox([
    widgets.HBox([tab1, tab2_container],layout=widgets.Layout(grid_gap = '20px 25px')),
    output_area
],layout=widgets.Layout(grid_gap = '20px 25px'))        
config_tab = widgets.Tab([
    tab
])
config_tab.set_title(0, "Data/Channel config")
display(config_tab)
with output_area:
    clear_output()
    print(f"Please complete and save this information...")


### Train Cellpose

Select which channels to use as inputs to Cellpose, confirm the model output name, then click **Train the Cellpose model**.

Training will:
1. Extract 2D planes from your 3D training and validation images (required by Cellpose)
2. Fine-tune the base Cellpose model (`cyto3` or `nuclei`) on your data
3. Save the trained model to the `models/` folder (created automatically in the same directory as this notebook)

In [ ]:
# Selector for inference on organoids
selector_widget = [widgets.Checkbox(value=False, description=lab) for lab in labels]
if (option_widget.value=='Organoid'):
    default_value='cellpose_organoids' 
elif (option_widget.value=='Immune cells/Other cell types'):
    default_value='cellpose_immune_other'

model_widget = widgets.Text(
    value=default_value,
    description='Basename for the '+option_widget.value+' model:',
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='auto')
)


load_button = widgets.Button(description="Train the Cellpose model", button_style='success',layout={'width': 'max-content'})
output_area = widgets.Output()

def on_load_clicked(b):
    with output_area:
        clear_output()
        global channels, model_path
        channels = [cb.description for cb in selector_widget if cb.value]
        assert len(channels)>0, f"You must use at least a channel for training the Cellpose model\n"
        model_str = model_widget.value+'__'+'-'.join(['channel'+str(i)+'-'+channels[i] for i in range(len(channels))])
        model_path = os.path.join('models',model_str)
        channels = [labels.index(channels[i]) for i in range(len(channels))]
        # Generate the 2D planes for training the network
        print(f"Generating images and labels for training Cellpose")
        generate_2D_planes(train_dir_picker.value, image_filter.value, mask_filter.value, dim_order, channels, anisotropy)
        print(f"✅ Generated images and labels for training Cellpose")
        print(f"Generating images and labels for validating Cellpose")
        generate_2D_planes(val_dir_picker.value, image_filter.value, mask_filter.value, dim_order, channels, anisotropy)
        print(f"✅ Generated images and labels for validating Cellpose")
        train_dir_cp=os.path.join(train_dir_picker.value,'2D_cellpose'+mask_filter.value)
        val_dir_cp=os.path.join(val_dir_picker.value,'2D_cellpose'+mask_filter.value)
        if (option_widget.value=='Organoid'):
            print(f"Training Cellpose Organoid model...")
            train_cellpose(train_dir_cp, val_dir_cp, len(channels), 400, 'cyto3', model_str)
            print(f"✅ Successfully trained Cellpose Organoid model")
        elif (option_widget.value=='Immune cells/Other cell types'):
            print(f"Training Cellpose Immune cells/Other cell types")
            train_cellpose(train_dir_cp, val_dir_cp, len(channels), 400, 'nuclei', model_str)
            print(f"✅ Successfully trained Cellpose Immune cells/Other cell types")
        print(f"\n📁 Model saved to: {model_path}")

load_button.on_click(on_load_clicked)
box= widgets.VBox([widgets.HTML("<b>Select the channels you want to use as Cellpose inputs:</b>"), 
                   *selector_widget, model_widget, load_button, output_area],
                             layout=widgets.Layout(grid_gap = '20px 25px'))
train_tab = widgets.Tab(children=[box], layout={'width': 'max-content'})
train_tab.set_title(0, 'Training Cellpose')
display(train_tab)


### Check examples of the validation data

In [ ]:
if (image_filter.value==''):
    list_of_val_images=glob.glob(os.path.join(val_dir_picker.value,'*'+mask_filter.value+'*'))
    list_of_val_images=[f.replace(mask_filter.value,image_filter.value) for f in list_of_val_images]
else:
    list_of_val_images=glob.glob(os.path.join(val_dir_picker.value,'*'+image_filter.value+'*'))
val_dropdown_widget = widgets.Dropdown(
    options=list_of_val_images,
    value=list_of_val_images[0],
    description="Sample:",
    layout=widgets.Layout(width="350px"),
    disabled=False,
)
model_path_picker = PathPicker(
    mode='file',
    start_dir='models',
    description='Model path:',
)

# Pre-fill with model_path if already set by training cell
if 'model_path' in dir():
    model_path_picker.value = str(model_path)

display(val_dropdown_widget, model_path_picker)
load_button = widgets.Button(description="Visualize the results", button_style='success',layout={'width': 'max-content'})
output_area = widgets.Output()

def channels_from_model_name(model_file, labels):
    """Recover the channel indices a model was trained on from its filename.

    Models are saved as `<basename>__channel0-<label>-channel1-<label>...`, so the
    labels encoded in the name are the ground truth for how many input channels the
    checkpoint expects. Returns None if the name does not carry that information.
    """
    name = os.path.basename(str(model_file))
    if '__' not in name:
        return None
    encoded = name.split('__', 1)[1]
    chans = []
    for part in encoded.split('-channel'):
        label = part.split('-', 1)[1] if '-' in part else None
        if label is None or label not in labels:
            return None
        chans.append(labels.index(label))
    return chans or None

def on_load_clicked(b):
    global channel_colors
    channel_colors=[]
    with output_area:
        clear_output()
        # Resolve model path from widget or global
        current_model_path = model_path_picker.value if model_path_picker.value else str(globals().get('model_path',''))
        assert current_model_path, "No model path set — please select a model file."
        # Work out which channels this model expects. Prefer the channels encoded in the
        # model filename (works even for a model picked from disk in a fresh kernel), then
        # the globals set by the training cell, and only then fall back to all channels.
        # NOTE: use globals(), not dir(): inside a function dir() lists *locals* only, so
        # 'channels' was never found and nchan silently defaulted to len(labels), which
        # crashes model loading with a state_dict size mismatch.
        current_channels = channels_from_model_name(current_model_path, labels)
        if current_channels is None:
            current_channels = globals().get('channels') or list(range(len(labels)))
        print(f"Using channels {current_channels} ({[labels[c] for c in current_channels]}) for inference")
        # Load and prepare the images
        img_val = io.imread(val_dropdown_widget.value)
        gt_val = io.imread(val_dropdown_widget.value.replace(image_filter.value,mask_filter.value))
        # Make the images to have channel axis first and z-axis second
        if (dim_order=='YXZ'):
            img_val = img_val.transpose(0,3,1,2)
            gt_val = gt_val.transpose(2,0,1)
        C, Z, Y, X = img_val.shape
        # Select the useful channels
        if (len(current_channels)>1):
            img_val = img_val[current_channels,...]
            nchan=len(current_channels)
            C=len(current_channels)
        else:
            # Models trained on <3 channels are built with nchan=2 (see train_cellpose),
            # so pad the single channel with zeros to match the checkpoint.
            img_val = np.concatenate((img_val[current_channels,...],np.zeros_like(img_val[current_channels,...])),axis=0)
            nchan=2
            C=1
        # Set the minimum size depending on the object to segment
        if (option_widget.value=='Organoid'):
            min_size=1000
        elif (option_widget.value=='Immune cells/Other cell types'):
            min_size=10
        # Determine computation device.
        torch_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        # Initialise Cellpose model.
        model = models.CellposeModel(
            pretrained_model=current_model_path,
            gpu=True,
            nchan=nchan,
            device=torch_device
        )
        # Cellpose expects channels first (C, Z, Y, X) with channel_axis=0, z_axis=1.
        mask_pred, *_ = model.eval(
            img_val,
            diameter=None,
            channels=None,
            min_size=min_size,
            do_3D=True,
            normalize=True,
            channel_axis=0,
            z_axis=1,
            anisotropy=anisotropy,
        )
        # Get channel colors — one per selected training channel, named from user input
        PALETTE = ['cyan', 'yellow', 'red', 'green', 'magenta', 'blue', 'orange', 'white']
        channel_colors = [PALETTE[i % len(PALETTE)] for i in range(len(current_channels))]
        # Launch viewer
        viewer = napari.Viewer()
        for ch in range(C):
            channel_data = img_val[ch]  # (Z, Y, X)
            viewer.add_image(
                channel_data,
                name=labels[current_channels[ch]],
                colormap=channel_colors[ch],
                scale=(1, 1, 1),
                blending="additive",
                channel_axis=None,
            )
        # Add all masks as separate label layers
        print(f"Adding ground-truth mask layer and prediction to viewer...")
        viewer.add_labels(gt_val, name='GT', scale=(1, 1, 1))
        viewer.add_labels(mask_pred, name='prediction', scale=(1, 1, 1))
        print("Launching Napari viewer...")
        napari.run()
load_button.on_click(on_load_clicked)
display(load_button, output_area)